# 02 – Conversion Prediction Modeling
**Digital Ad Campaign Measurement & Attribution Framework**

Trains and evaluates XGBoost, Random Forest, and MLP classifiers to predict
conversion probability. All experiments are tracked with **MLflow**.

Targeting scenarios:
- **Addressable** – user-level features (recency, frequency, prior_clicks)
- **Cohort-based** – cohort_size, aggregated signals
- **Contextual**  – context_score, page signals only


In [ ]:
import os, sys, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.join('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import mlflow

from src.preprocessing import load_raw, build_feature_matrix
from src.models.train  import train_all_models, prepare_xy
from src.models.evaluate import (
    plot_roc, plot_pr, plot_calibration, plot_lift,
    plot_feature_importance, comparison_table,
)

In [ ]:
df_raw = load_raw()
df_enc, encoders, scaler = build_feature_matrix(df=df_raw)
print(f'Feature matrix: {df_enc.shape}')

## 1. Train Models (all targeting strategies combined)

In [ ]:
mlflow.set_tracking_uri('file:../mlruns')
models = train_all_models(df_enc)
print('Models trained:', list(models.keys()))

## 2. Evaluation on Hold-out Test Set

In [ ]:
X, y       = prepare_xy(df_enc)
split_idx  = int(len(df_enc) * 0.80)
X_test     = X.iloc[split_idx:]
y_test     = y[split_idx:]

comp = comparison_table(models, X_test, y_test)
comp

In [ ]:
plot_roc(models, X_test, y_test, save=False)
plt.show()

In [ ]:
plot_pr(models, X_test, y_test, save=False)
plt.show()

In [ ]:
plot_calibration(models, X_test, y_test, save=False)
plt.show()

In [ ]:
plot_lift(models['XGBoost'], X_test, y_test, model_name='XGBoost', save=False)
plt.show()

## 3. Feature Importance

In [ ]:
plot_feature_importance(models['XGBoost'], list(X.columns), model_name='XGBoost', save=False)
plt.show()

## 4. SHAP Explainability

In [ ]:
from src.explainability.shap_analysis import compute_shap_values, plot_summary, plot_waterfall, top_features_table

sv = compute_shap_values(models['XGBoost'], X, sample_size=2000)
plot_summary(sv, model_name='XGBoost', save=False)
plt.show()

In [ ]:
plot_waterfall(sv, sample_idx=0, model_name='XGBoost', save=False)
plt.show()

In [ ]:
top_features_table(sv, top_n=15)

## 5. MLflow Experiment Runs

In [ ]:
# View experiment runs programmatically
runs = mlflow.search_runs(experiment_names=['ad-conversion-prediction'])
cols = ['tags.mlflow.runName','metrics.roc_auc','metrics.pr_auc','metrics.f1','metrics.brier']
cols = [c for c in cols if c in runs.columns]
runs[cols].sort_values('metrics.roc_auc', ascending=False).head(10)